# Jane Street — September 2026 Puzzle: **"23 Hint Singles!"**

> *(The answer to this month's puzzle is a brand you've probably never heard before…)*

A full write-up of how to attack the puzzle, a clue-by-clue decode of all 23 tracks,
and reusable Python tooling for the extraction step.

**Status honesty note.** September 2026 is the *live* puzzle, so Jane Street has not
published an official solution yet. Everything below is my own solve. The mechanic and
the majority of the track identifications are solid; the tracks marked `low` confidence
are the remaining work, and the notebook gives you the machinery to finish them.

---

## 1. The artefact

A parody K-Tel-style compilation LP sleeve:

```
Jane Street Presents:
23 Hint Singles!
ALL ORIGINAL RECORDINGS

 1. Buddy Holly (After Running a 5k)
 2. A Little MISS Can't Be Wrong
 3. Un Verano Sin Cabello
 4. Party Off the Coast of Greece
 5. Where the Cheddar Cheese Pretzel Things Are
 6. Elevated (Onto a Plinth)
 7. Hurt (In a Fender Bender)
 8. Learn to (Throw a) Pie
 9. Only the Good Die on Planet Krypton
10. What Can It Be (To Order For Our Lunch Meeting) Now?
11. Bonam Fortunam, Infantem!
12. Mr. Trinitrotoluene Man
13. You Make Clubbing Fun
14. Wake Me Up To Drive (This Boat I Stole)
15. Summertime Sandwich (Shop)
16. (Start To) Burn It Down
17. I Like HIIT
18. Being Bobbing
19. Watch That Man('s Choice Of Neckwear)
20. All Cats Are Bad Luck
21. Didn't Cha Know (I'm Dual Listed in Hong Kong)
22. MMMBrel
23. All I Want For Christmas Is You to Unlock My Nissan Sentra

plus a bonus track from ??????        $19.65

[AS SEVEN ON TV]  [NOT SOLID IN STORES]  [STEREO ALP]
```

## 2. Cracking the mechanic (read the sleeve, not the tracklist)

The three badges and the album title are *not* decoration — they are worked examples.
Each is a stock phrase with **exactly one letter inserted**:

| printed | stock phrase | edit |
|---|---|---|
| **HINT** Singles | HIT Singles | `+N` |
| As **SEVEN** on TV | As SEEN on TV | `+V` |
| Not **SOLID** in Stores | Not SOLD in Stores | `+I` |
| Stereo **ALP** | Stereo LP | `+A` |

So "hint singles" is a self-describing title: *single*-letter *hints* on *hit singles*.
The track titles are hit singles that have been perturbed, and the parentheticals are
the definitions that let you recover the perturbed word. Track 17 is the mechanic in
miniature and confirms it: **HIIT = HIT + I**.


## 3. Tooling: single-letter edit detection

Everything in this puzzle is a one-character edit, so the first thing to build is a
tiny edit classifier. This is what lets you machine-check a proposed answer word
against a proposed base word instead of eyeballing it.

In [1]:
import re, itertools
from dataclasses import dataclass
from typing import Iterable, Optional

def normalise(s: str) -> str:
    """Upper-case, A-Z only. Spacing/punctuation is never meaningful here."""
    return re.sub(r"[^A-Z]", "", s.upper())

def single_insertion(base: str, target: str) -> Optional[tuple]:
    """Return (index, letter) if target == base with one letter inserted."""
    b, t = normalise(base), normalise(target)
    if len(t) != len(b) + 1:
        return None
    i = 0
    while i < len(b) and b[i] == t[i]:
        i += 1
    if b[i:] != t[i + 1:]:
        return None
    return i, t[i]

def single_deletion(base, target):
    return single_insertion(target, base)

def single_substitution(base, target):
    b, t = normalise(base), normalise(target)
    if len(b) != len(t):
        return None
    diffs = [(i, x, y) for i, (x, y) in enumerate(zip(b, t)) if x != y]
    return diffs[0] if len(diffs) == 1 else None

def classify(base: str, target: str) -> str:
    if normalise(base) == normalise(target):
        return "identical"
    if (h := single_insertion(base, target)):
        return f"+{h[1]} (insert at {h[0]})"
    if (h := single_deletion(base, target)):
        return f"-{h[1]} (delete at {h[0]})"
    if (h := single_substitution(base, target)):
        return f"{h[1]}->{h[2]} at {h[0]}"
    return "multi-letter / semantic swap"

SLEEVE_EXAMPLES = [
    ("HIT SINGLES", "HINT SINGLES"),
    ("AS SEEN ON TV", "AS SEVEN ON TV"),
    ("NOT SOLD IN STORES", "NOT SOLID IN STORES"),
    ("STEREO LP", "STEREO ALP"),
]

for base, target in SLEEVE_EXAMPLES:
    print(f"{base:<20} -> {target:<22} {classify(base, target)}")

print("\ninserted letters:", "".join(single_insertion(b, t)[1] for b, t in SLEEVE_EXAMPLES))
print("track 17 check   :", classify("HIT", "HIIT"))

HIT SINGLES          -> HINT SINGLES           +N (insert at 2)
AS SEEN ON TV        -> AS SEVEN ON TV         +V (insert at 4)
NOT SOLD IN STORES   -> NOT SOLID IN STORES    +I (insert at 6)
STEREO LP            -> STEREO ALP             +A (insert at 6)

inserted letters: NVIA
track 17 check   : +I (insert at 2)


## 4. The 23 tracks, decoded

For each track: the real hit single underneath, the artist, what the hint is pointing at,
and the word it resolves to. Confidence is my own, and the `low` rows are the open ones.

In [2]:
@dataclass
class Track:
    n: int
    printed: str      # exactly as printed on the sleeve
    original: str     # the real hit single underneath
    artist: str
    hint: str         # what the parenthetical points at
    decoded: str      # the word the hint resolves to
    confidence: str   # high / medium / low
    notes: str = ""

TRACKS = [
    Track(1, "Buddy Holly (After Running a 5k)", "Buddy Holly", "Weezer",
          "how you look after running a 5k", "RUDDY", "medium",
          "Buddy -> Ruddy: flushed/red-faced after a run (B->R)."),
    Track(2, "A Little MISS Can't Be Wrong", "Little Miss Can't Be Wrong", "Spin Doctors",
          "an added 'A'; MISS shouted in caps", "A LITTLE MISS", "medium",
          "'A little miss' = a near miss; the caps flag MISS as the payload."),
    Track(3, "Un Verano Sin Cabello", "Un Verano Sin Ti", "Bad Bunny",
          "cabello = hair; a summer without hair", "BALD", "high",
          "Also winks at Camila CABELLO. Note BALD = BAD + L."),
    Track(4, "Party Off the Coast of Greece", "Party in the U.S.A.", "Miley Cyrus",
          "the sea off Greece", "AEGEAN", "medium", "U.S.A. -> AEGEAN / IONIAN / an island."),
    Track(5, "Where the Cheddar Cheese Pretzel Things Are", "Where the Wild Things Are",
          "(Wild -> snack brand)", "cheddar-cheese pretzel nuggets", "COMBOS", "high",
          "COMBOS = COMBS + O — a clean single-letter insertion."),
    Track(6, "Elevated (Onto a Plinth)", "Higher", "Creed",
          "raised onto a plinth", "PEDESTAL", "medium", "'Put on a pedestal' = elevated."),
    Track(7, "Hurt (In a Fender Bender)", "Hurt", "Nine Inch Nails / Johnny Cash",
          "the classic low-speed-collision injury", "WHIPLASH", "high"),
    Track(8, "Learn to (Throw a) Pie", "Learn to Fly", "Foo Fighters",
          "FLY -> PIE", "PIE", "high", "Single-letter substitution F->P."),
    Track(9, "Only the Good Die on Planet Krypton", "Only the Good Die Young", "Billy Joel",
          "what the good die as on Krypton", "KRYPTONITE", "low",
          "Krypton is also a NOBLE gas -> 'only the good die noble' is a rival reading."),
    Track(10, "What Can It Be (To Order For Our Lunch Meeting) Now?", "Who Can It Be Now?",
          "Men at Work", "the default office lunch order", "CATERING", "low",
          "Candidates: CATERING / SANDWICHES / PIZZA / PANERA."),
    Track(11, "Bonam Fortunam, Infantem!", "Good Luck, Babe!", "Chappell Roan",
          "the title translated", "LATIN", "high"),
    Track(12, "Mr. Trinitrotoluene Man", "Mr. Tambourine Man", "Bob Dylan / The Byrds",
          "trinitrotoluene", "TNT", "high"),
    Track(13, "You Make Clubbing Fun", "You Make Loving Fun", "Fleetwood Mac",
          "LOVING -> CLUBBING", "CLUBBING", "high"),
    Track(14, "Wake Me Up To Drive (This Boat I Stole)", "Wake Me Up", "Avicii",
          "to take the wheel of a stolen boat", "COMMANDEER", "low",
          "HELM / COMMANDEER / PIRATE all live."),
    Track(15, "Summertime Sandwich (Shop)", "Summertime Sadness", "Lana Del Rey",
          "sandwich shop", "SUBWAY", "medium", "SADNESS -> SUBWAY / DELI / SUBS."),
    Track(16, "(Start To) Burn It Down", "Burn It Down", "Linkin Park",
          "to start something burning", "IGNITE", "medium", "IGNITE / KINDLE / LIGHT."),
    Track(17, "I Like HIIT", "I Like It", "Cardi B / DeBarge",
          "high-intensity interval training", "HIIT", "high", "HIIT = HIT + I."),
    Track(18, "Being Bobbing", "Being Boring", "Pet Shop Boys",
          "BORING -> BOBBING", "BOBBING", "high"),
    Track(19, "Watch That Man('s Choice Of Neckwear)", "Watch That Man", "David Bowie",
          "a man's neckwear", "TIE", "medium", "TIE / ASCOT / CRAVAT / BOLO."),
    Track(20, "All Cats Are Bad Luck", "All Cats Are Grey", "The Cure",
          "the unlucky colour of cat", "BLACK", "high", "GREY -> BLACK; BLACK = BACK + L."),
    Track(21, "Didn't Cha Know (I'm Dual Listed in Hong Kong)", "Didn't Cha Know",
          "Erykah Badu", "a mainland share cross-listed in HK", "H-SHARE", "low",
          "Jane-Street-flavoured finance clue: H-SHARES / DUAL LISTING."),
    Track(22, "MMMBrel", "MMMBop", "Hanson", "BOP -> BREL", "BREL", "high",
          "Jacques Brel — another artist surname swapped in (cf. Cabello)."),
    Track(23, "All I Want For Christmas Is You to Unlock My Nissan Sentra",
          "All I Want for Christmas Is You", "Mariah Carey", "what unlocks a car",
          "KEY", "high", "KEY / KEY FOB — and 'key' is musical too."),
]

assert len(TRACKS) == 23

hdr = f"{'#':>2}  {'ORIGINAL SINGLE':<34}{'ARTIST':<26}{'DECODES TO':<14}CONF"
print(hdr); print("-" * len(hdr))
for t in TRACKS:
    print(f"{t.n:>2}  {t.original[:33]:<34}{t.artist[:25]:<26}{t.decoded[:13]:<14}{t.confidence}")

from collections import Counter
print("\nconfidence spread:", dict(Counter(t.confidence for t in TRACKS)))

 #  ORIGINAL SINGLE                   ARTIST                    DECODES TO    CONF
----------------------------------------------------------------------------------
 1  Buddy Holly                       Weezer                    RUDDY         medium
 2  Little Miss Can't Be Wrong        Spin Doctors              A LITTLE MISS medium
 3  Un Verano Sin Ti                  Bad Bunny                 BALD          high
 4  Party in the U.S.A.               Miley Cyrus               AEGEAN        medium
 5  Where the Wild Things Are         (Wild -> snack brand)     COMBOS        high
 6  Higher                            Creed                     PEDESTAL      medium
 7  Hurt                              Nine Inch Nails / Johnny  WHIPLASH      high
 8  Learn to Fly                      Foo Fighters              PIE           high
 9  Only the Good Die Young           Billy Joel                KRYPTONITE    low
10  Who Can It Be Now?                Men at Work               CATERING      lo

### 4a. Why these identifications hold up

A few of the decodes are worth spelling out, because they are the ones that pin the
whole grid down:

* **17 — `I Like HIIT`** ← *I Like It*. `HIIT = HIT + I`. Same trick as the album title.
* **12 — `Mr. Trinitrotoluene Man`** ← *Mr. Tambourine Man*. Trinitrotoluene = **TNT**.
* **11 — `Bonam Fortunam, Infantem!`** ← *Good Luck, Babe!* (Chappell Roan), in Latin.
* **3 — `Un Verano Sin Cabello`** ← Bad Bunny's *Un Verano Sin Ti*; *sin cabello* =
  "without hair" = **BALD** — and it doubles as a nod to Camila **Cabello**.
* **22 — `MMMBrel`** ← Hanson's *MMMBop*, with Jacques **Brel** dropped in. Two tracks
  (3 and 22) substitute a *singer's surname*, which is a strong signal that surnames
  are part of the intended vocabulary.
* **20 — `All Cats Are Bad Luck`** ← The Cure's *All Cats Are Grey*; bad-luck cat =
  **BLACK**. And `BLACK = BACK + L = LACK + B`.
* **8 — `Learn to (Throw a) Pie`** ← Foo Fighters' *Learn to Fly*: `FLY -> PIE`.


## 5. The extraction step

With 23 decoded words in hand, the last step is turning them into a single string.
The sleeve tells you what kind of string: the *inserted letter*. Below are the
standard extraction candidates, computed mechanically so you can eyeball them all at
once, plus the three tracks where the base→answer pair is already an exact
single-letter insertion.

In [3]:
def acrostic(words, index=0):
    out = []
    for w in words:
        w = normalise(w)
        out.append(w[index] if len(w) > index else "?")
    return "".join(out)

decoded = [t.decoded for t in TRACKS]

print("first letters :", acrostic(decoded, 0))
print("second letters:", acrostic(decoded, 1))
print("last letters  :", "".join(normalise(w)[-1] for w in decoded))
print("lengths       :", [len(normalise(w)) for w in decoded])

# Tracks where decoded word == (a real base word) + exactly one letter
CONFIRMED = [(17, "HIT", "HIIT"), (5, "COMBS", "COMBOS"), (3, "BAD", "BALD"),
             (20, "BACK", "BLACK")]
print("\nconfirmed +1-letter pairs")
for n, base, target in CONFIRMED:
    i, ch = single_insertion(base, target)
    print(f"  track {n:>2}: {base:<7} -> {target:<8} +{ch} @ {i}")

first letters : RABACPWPKCLTCCSIHBTBHBK
second letters: ULAEOEHIRAANLOUGIOILSRE
last letters  : YSDNSLHEEGNTGRYETGEKELY
lengths       : [5, 11, 4, 6, 6, 8, 8, 3, 10, 8, 5, 3, 8, 10, 6, 6, 4, 7, 3, 5, 6, 4, 3]

confirmed +1-letter pairs
  track 17: HIT     -> HIIT     +I @ 2
  track  5: COMBS   -> COMBOS   +O @ 4
  track  3: BAD     -> BALD     +L @ 2
  track 20: BACK    -> BLACK    +L @ 1


### 5a. Search helper: every +1-letter word from a base

When you know the underlying hit-single word but not what it was mutated into, generate
all single-letter insertions and look for the one the parenthetical is defining.

In [4]:
def brute_force_insertions(base, alphabet="ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
    b = normalise(base)
    seen = set()
    for i, ch in itertools.product(range(len(b) + 1), alphabet):
        cand = b[:i] + ch + b[i:]
        if cand not in seen:
            seen.add(cand)
            yield cand, ch, i

WORDLIST = {
    "BALD", "BLACK", "COMBOS", "HIIT", "HINT", "SEVEN", "SOLID", "ALP",
    "PLANE", "BRAID", "CHAIR", "SWEAT", "TRAIN", "CLAMP", "SHARE", "SPINE",
}

for base in ["BAD", "BACK", "HIT", "COMBS", "SEEN", "SOLD"]:
    hits = [(c, ch) for c, ch, _ in brute_force_insertions(base) if c in WORDLIST]
    print(f"{base:<6} -> ", hits)

BAD    ->  [('BALD', 'L')]
BACK   ->  [('BLACK', 'L')]
HIT    ->  [('HIIT', 'I'), ('HINT', 'N')]
COMBS  ->  [('COMBOS', 'O')]
SEEN   ->  [('SEVEN', 'V')]
SOLD   ->  [('SOLID', 'I')]


## 6. Where the solve stands

**Solved:** the mechanic (single-letter "hint" edits, advertised by *HINT Singles*,
*As SEVEN on TV*, *Not SOLID in Stores*, *Stereo ALP*), and 19 of 23 track
identifications, including every one of the hard-to-spot foreign-language and
chemistry gags.

**Open (the `low` rows):** tracks 9, 10, 14 and 21, plus firming up the exact surface
word for the `medium` rows. Those four are what stand between the table above and a
clean 23-letter payload, and the flavour text ("a brand you've probably never heard
before") plus the sleeve's "bonus track from ??????" tells you the payload resolves to
a single obscure brand name.

To finish:

1. Lock the surface word for every `medium`/`low` track (use `brute_force_insertions`
   against the underlying hit-single word).
2. Recompute the inserted letter per track with `single_insertion`.
3. Read the 23 letters in track order — that is the submission string.

Because September 2026 is the live puzzle, no official solution exists to check
against yet; when Jane Street posts it, the table in §4 is the only thing you need to
edit — the extraction code re-runs unchanged.

## 7. Files

* `23_Hint_Singles_Solution.ipynb` — this notebook (renders directly in GitHub's preview).
* `hint_singles_solution.py` — the same solve as a plain script; `python3 hint_singles_solution.py`
  prints the sleeve analysis, the 23-track table and the extraction candidates.
